# 04 — Build gold
Aggregated tables answering specific questions. Each is independent —
reads silver, writes its own output.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

flights = spark.read.table("flights.silver.flights")
airlines = spark.read.table("flights.silver.airlines")

# Airline reliability
airline_reliability = (
    flights
    .filter(F.col("is_cancelled") == False)
    .groupBy("airline_code")
    .agg(
        F.count("*").alias("total_flights"),
        F.sum(F.col("dep_delayed_15").cast("int")).alias("delayed_flights")
    )
    .withColumn("pct_delayed",
        F.round(F.col("delayed_flights") / F.col("total_flights") * 100, 1))
    .join(airlines, "airline_code", "left")
    .select("airline_name", "total_flights", "delayed_flights", "pct_delayed")
    .orderBy("pct_delayed")
)

airline_reliability.write.mode("overwrite").saveAsTable("flights.gold.airline_reliability")

# Route reliability — routes with at least 500 flights
route_reliability = (
    flights
    .filter(F.col("is_cancelled") == False)
    .groupBy("origin_airport", "dest_airport")
    .agg(
        F.count("*").alias("total_flights"),
        F.round(F.avg(F.col("dep_delayed_15").cast("int")) * 100, 1).alias("pct_delayed")
    )
    .filter(F.col("total_flights") >= 500)
    .orderBy(F.col("pct_delayed").desc())
)

route_reliability.write.mode("overwrite").saveAsTable("flights.gold.route_reliability")

# Airline ranking within each route
route_airline = (
    flights
    .filter(F.col("is_cancelled") == False)
    .groupBy("origin_airport", "dest_airport", "airline_code")
    .agg(
        F.count("*").alias("total_flights"),
        F.round(F.avg(F.col("dep_delayed_15").cast("int")) * 100, 1).alias("pct_delayed")
    )
    .filter(F.col("total_flights") >= 100)
)

route_window = Window.partitionBy("origin_airport", "dest_airport").orderBy("pct_delayed")

ranked = route_airline.withColumn("rank", F.row_number().over(route_window))

ranked.write.mode("overwrite").saveAsTable("flights.gold.route_airline_ranked")

In [0]:
# Monthly delays — the seasonal pattern
monthly_delays = (
    flights
    .filter(F.col("is_cancelled") == False)
    .groupBy(F.date_format("flight_date", "yyyy-MM").alias("month"))
    .agg(
        F.count("*").alias("total_flights"),
        F.round(F.avg(F.col("dep_delayed_15").cast("int")) * 100, 1).alias("pct_delayed")
    )
    .orderBy("month")
)

monthly_delays.write.mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("flights.gold.monthly_delays")
spark.read.table("flights.gold.monthly_delays").show(12)

In [0]:
# Delay causes by month — why is July worst?
delay_causes_monthly = (
    flights
    .filter(F.col("is_cancelled") == False)
    .groupBy(F.date_format("flight_date", "yyyy-MM").alias("month"))
    .agg(
        F.round(F.sum("carrier_delay_min") / 60, 0).alias("carrier_hours"),
        F.round(F.sum("weather_delay_min") / 60, 0).alias("weather_hours"),
        F.round(F.sum("nas_delay_min") / 60, 0).alias("atc_hours"),
        F.round(F.sum("security_delay_min") / 60, 0).alias("security_hours"),
        F.round(F.sum("late_aircraft_delay_min") / 60, 0).alias("late_aircraft_hours")
    )
    .orderBy("month")
)

delay_causes_monthly.write.mode("overwrite").saveAsTable("flights.gold.delay_causes_monthly")

In [0]:
# Delays by day of week — are Fridays really worse?
delays_by_dayofweek = (
    flights
    .filter(F.col("is_cancelled") == False)
    .groupBy(
        F.dayofweek("flight_date").alias("day_number"),
        F.date_format("flight_date", "EEEE").alias("day_name")
    )
    .agg(
        F.count("*").alias("total_flights"),
        F.round(F.avg(F.col("dep_delayed_15").cast("int")) * 100, 1).alias("pct_delayed")
    )
    .orderBy("day_number")
)

delays_by_dayofweek.write.mode("overwrite").saveAsTable("flights.gold.delays_by_dayofweek")

In [0]:
# Delays by hour of day — are morning flights really better?
delays_by_hour = (
    flights
    .filter(F.col("is_cancelled") == False)
    .filter(F.col("scheduled_dep_time").isNotNull())
    .withColumn("dep_hour", F.substring("scheduled_dep_time", 1, 2).cast("int"))
    .groupBy("dep_hour")
    .agg(
        F.count("*").alias("total_flights"),
        F.round(F.avg(F.col("dep_delayed_15").cast("int")) * 100, 1).alias("pct_delayed")
    )
    .filter(F.col("total_flights") >= 1000)
    .orderBy("dep_hour")
)

delays_by_hour.write.mode("overwrite").saveAsTable("flights.gold.delays_by_hour")

In [0]:
# Airport reliability — which airports are worst for departures? (10,000+ flights)
airports = spark.read.table("flights.silver.airports")

airport_reliability = (
    flights
    .filter(F.col("is_cancelled") == False)
    .groupBy("origin_airport")
    .agg(
        F.count("*").alias("total_flights"),
        F.round(F.avg(F.col("dep_delayed_15").cast("int")) * 100, 1).alias("pct_delayed"),
        F.round(F.avg("dep_delay_min"), 1).alias("avg_delay_min")
    )
    .filter(F.col("total_flights") >= 10000)
    .join(airports, F.col("origin_airport") == F.col("airport_code"), "left")
    .select("origin_airport", "city_name", "state_name", "total_flights", "pct_delayed", "avg_delay_min")
    .orderBy(F.col("pct_delayed").desc())
)

airport_reliability.write.mode("overwrite").saveAsTable("flights.gold.airport_reliability")

In [0]:
# Cancellation rates — how often, by which airline, and why?
cancellation_rates = (
    flights
    .groupBy("airline_code")
    .agg(
        F.count("*").alias("total_flights"),
        F.sum(F.col("is_cancelled").cast("int")).alias("cancelled_flights"),
        F.round(F.avg(F.col("is_cancelled").cast("int")) * 100, 2).alias("pct_cancelled"),
        F.sum(F.when(F.col("cancellation_code") == "A", 1).otherwise(0)).alias("carrier_cause"),
        F.sum(F.when(F.col("cancellation_code") == "B", 1).otherwise(0)).alias("weather_cause"),
        F.sum(F.when(F.col("cancellation_code") == "C", 1).otherwise(0)).alias("atc_cause"),
        F.sum(F.when(F.col("cancellation_code") == "D", 1).otherwise(0)).alias("security_cause")
    )
    .join(airlines, "airline_code", "left")
    .select("airline_name", "total_flights", "cancelled_flights", "pct_cancelled",
            "carrier_cause", "weather_cause", "atc_cause", "security_cause")
    .orderBy("pct_cancelled")
)

cancellation_rates.write.mode("overwrite").saveAsTable("flights.gold.cancellation_rates")

In [0]:
# Arrival delays by day of week — compare with the departure version
arrival_delays_by_dayofweek = (
    flights
    .filter(F.col("is_cancelled") == False)
    .filter(F.col("is_diverted") == False)
    .groupBy(
        F.dayofweek("flight_date").alias("day_number"),
        F.date_format("flight_date", "EEEE").alias("day_name")
    )
    .agg(
        F.count("*").alias("total_flights"),
        F.round(F.avg(F.col("arr_delayed_15").cast("int")) * 100, 1).alias("pct_delayed")
    )
    .orderBy("day_number")
)

arrival_delays_by_dayofweek.write.mode("overwrite").saveAsTable("flights.gold.arrival_delays_by_dayofweek")

In [0]:
# Arrival delays by scheduled arrival hour — does the cascade show on arrivals too?
arrival_delays_by_hour = (
    flights
    .filter(F.col("is_cancelled") == False)
    .filter(F.col("is_diverted") == False)
    .filter(F.col("scheduled_arr_time").isNotNull())
    .withColumn("arr_hour", F.substring("scheduled_arr_time", 1, 2).cast("int"))
    .groupBy("arr_hour")
    .agg(
        F.count("*").alias("total_flights"),
        F.round(F.avg(F.col("arr_delayed_15").cast("int")) * 100, 1).alias("pct_delayed")
    )
    .filter(F.col("total_flights") >= 1000)
    .orderBy("arr_hour")
)

arrival_delays_by_hour.write.mode("overwrite").saveAsTable("flights.gold.arrival_delays_by_hour")

In [0]:
# Airport arrival reliability — which airports are worst to land at? (10,000+ flights)
airport_arrival_reliability = (
    flights
    .filter(F.col("is_cancelled") == False)
    .filter(F.col("is_diverted") == False)
    .groupBy("dest_airport")
    .agg(
        F.count("*").alias("total_flights"),
        F.round(F.avg(F.col("arr_delayed_15").cast("int")) * 100, 1).alias("pct_delayed"),
        F.round(F.avg("arr_delay_min"), 1).alias("avg_delay_min")
    )
    .filter(F.col("total_flights") >= 10000)
    .join(airports, F.col("dest_airport") == F.col("airport_code"), "left")
    .select("dest_airport", "city_name", "state_name",
            "total_flights", "pct_delayed", "avg_delay_min")
    .orderBy(F.col("pct_delayed").desc())
)

airport_arrival_reliability.write.mode("overwrite").saveAsTable("flights.gold.airport_arrival_reliability")

In [0]:
spark.sql("SHOW TABLES IN flights.gold").show()

In [0]:
# Reconcile Southwest count: silver vs gold
silver_wn = spark.read.table("flights.silver.flights") \
    .filter(F.col("airline_code") == "WN") \
    .filter(F.col("is_cancelled") == False) \
    .count()

gold_wn = spark.read.table("flights.gold.airline_reliability") \
    .filter(F.col("airline_name") == "Southwest Airlines") \
    .select("total_flights").collect()[0][0]

print(f"Southwest — silver: {silver_wn}, gold: {gold_wn}")
assert silver_wn == gold_wn, \
    f"Southwest count mismatch: silver={silver_wn}, gold={gold_wn}"